# init-process-group-nccl — faded example 2: Fill the finally destroy in the session

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `init-process-group-nccl`. Running the beacon reports progress on the `Distributed: init_process_group nccl` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: init_process_group nccl` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`init-process-group-nccl`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "init-process-group-nccl"
DD_SUBTOPIC = "Distributed: init_process_group nccl"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In the distributed session context manager, the `destroy_process_group` call belongs in a `finally` block so cleanup runs even when the body raises, preventing a leaked communicator.

## Faded exercise 2

Implement `dist_session` as a `@contextlib.contextmanager`. Init is given; wrap the `yield` in `try/finally` so destroy always runs. Complete the blanked `finally` clause that destroys the group.

**Fill in:** the finally block that calls dist_module.destroy_process_group unconditionally

In [ ]:
import os
import contextlib

@contextlib.contextmanager
def dist_session(rank, world_size, port, dist_module, backend='gloo'):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist_module.init_process_group(backend=backend, rank=rank, world_size=world_size)
    try:
        yield
    finally:
        raise NotImplementedError()  # TODO: the finally block that calls dist_module.destroy_process_group unconditionally

class MockDist:
    def __init__(self):
        self.destroys = 0
    def init_process_group(self, **kw):
        pass
    def destroy_process_group(self):
        self.destroys += 1

m = MockDist()
with dist_session(0, 1, 29514, m):
    pass
print(m.destroys)


def _test():
    # normal path: destroy called exactly once
    m = MockDist()
    with dist_session(0, 1, 29514, m):
        pass
    assert m.destroys == 1, m.destroys
    # exception path: destroy still called exactly once
    m2 = MockDist()
    raised = False
    try:
        with dist_session(0, 1, 29514, m2):
            raise ValueError('boom')
    except ValueError:
        raised = True
    assert raised
    assert m2.destroys == 1, m2.destroys


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import os
import contextlib

@contextlib.contextmanager
def dist_session(rank, world_size, port, dist_module, backend='gloo'):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist_module.init_process_group(backend=backend, rank=rank, world_size=world_size)
    try:
        yield
    finally:
        dist_module.destroy_process_group()

class MockDist:
    def __init__(self):
        self.destroys = 0
    def init_process_group(self, **kw):
        pass
    def destroy_process_group(self):
        self.destroys += 1

m = MockDist()
with dist_session(0, 1, 29514, m):
    pass
print(m.destroys)
```
</details>